# Free Manga Translator — Kaggle GPU backend

Runs the 8-step manga translation pipeline on Kaggle's free NVIDIA T4 GPU and exposes it over an HTTPS tunnel so your Chrome extension (running locally in your browser) can reach it.

**Full documentation:** `docs/KAGGLE_DEPLOYMENT.md` in this repository — read it before your first run. This notebook mirrors that document's cell-by-cell walkthrough exactly.

**Before running anything:**
1. Side panel → Accelerator → **GPU T4 x2**.
2. Side panel → Internet → **On**.
3. Add-ons → Data → attach your private `fmt-core-pipeline` dataset.
4. Add-ons → Secrets → attach `FMT_ENV_B64`, `NGROK_AUTHTOKEN`, `FMT_AUTH_TOKEN` (and `GH_PAT` only if you're using the git-clone path instead of a dataset).
5. Edit `STATIC_DOMAIN` in cell 6 below to the domain you claimed in your ngrok dashboard.

## Cell 1 — environment sanity check
Confirms the GPU is actually visible and the dataset is attached before spending any notebook quota.

In [ ]:
import subprocess, os

assert os.path.isdir("/kaggle/input/fmt-core-pipeline"), (
    "Dataset not attached -- Add-ons -> Data -> attach your fmt-core-pipeline dataset"
)
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    capture_output=True, text=True,
).stdout)

## Cell 2 — copy code + install dependencies
`/kaggle/input` is read-only, so the code is copied to `/kaggle/working` first. Kaggle's base image usually already ships a recent torch build -- skip the pinned CUDA wheel reinstall when it already matches, since that install alone can take several minutes.

In [ ]:
import shutil, subprocess, sys
import torch

shutil.copytree(
    "/kaggle/input/fmt-core-pipeline/core_pipeline",
    "/kaggle/working/core_pipeline",
    dirs_exist_ok=True,
)
%cd /kaggle/working/core_pipeline

need_torch_reinstall = not (torch.__version__.startswith("2.6") and torch.cuda.is_available())
if need_torch_reinstall:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "torch==2.6.0", "torchvision==0.21.0",
         "--index-url", "https://download.pytorch.org/whl/cu124"],
        check=True,
    )
else:
    print(f"Reusing preinstalled torch {torch.__version__} (CUDA available: {torch.cuda.is_available()})")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "python/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "backend_api/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok"], check=True)
print("Dependencies installed.")

## Cell 3 — decode secrets into environment
Reads `FMT_ENV_B64` from Kaggle Secrets and decodes it back into a real `.env` file, then points the backend at it via `FMT_ENV_FILE` (the override `api_manager.py`'s `_load_env` checks first). `FMT_AUTH_TOKEN` is set directly as a process environment variable. **Nothing here is ever printed.**

In [ ]:
import base64, os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
env_bytes = base64.b64decode(secrets.get_secret("FMT_ENV_B64"))
env_path = "/kaggle/working/fmt.env"
with open(env_path, "wb") as f:
    f.write(env_bytes)
os.chmod(env_path, 0o600)

os.environ["FMT_ENV_FILE"] = env_path
os.environ["FMT_AUTH_TOKEN"] = secrets.get_secret("FMT_AUTH_TOKEN")
print("Secrets loaded (values not shown).")

## Cell 4 — launch the backend
Runs uvicorn the same way `start_backend.ps1` does locally (same working directory, same module path), as a background subprocess. Bind stays on `127.0.0.1` -- the tunnel client in cell 6 runs on this same VM and reaches it over loopback, so there's never a reason to bind wider here.

In [ ]:
import subprocess, sys

backend_log = open("/kaggle/working/backend.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend_api.app.main:app",
     "--host", "127.0.0.1", "--port", "8766"],
    cwd="/kaggle/working/core_pipeline",
    stdout=backend_log, stderr=subprocess.STDOUT,
)
print(f"Backend starting, pid={backend_proc.pid}. Logs: /kaggle/working/backend.log")

## Cell 5 — wait for health + force warmup
Polls `/v1/health` until the process accepts connections, then forces `/v1/warmup` and polls until it reports `pass`. See `docs/KAGGLE_DEPLOYMENT.md` §7 for realistic timing on a cold vs. warm cache.

In [ ]:
import time, urllib.request, json

def get_json(url, method="GET"):
    req = urllib.request.Request(
        url, method=method,
        headers={"X-Fmt-Client": "free-manga-translator-extension"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)

for _ in range(30):
    try:
        get_json("http://127.0.0.1:8766/v1/health")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not come up -- check /kaggle/working/backend.log")

get_json("http://127.0.0.1:8766/v1/warmup", method="POST")
for _ in range(60):
    status = get_json("http://127.0.0.1:8766/v1/health")["warmup"]["status"]
    print("warmup:", status)
    if status in ("pass", "fail"):
        break
    time.sleep(5)

## Cell 6 — open the tunnel
Uses `pyngrok` with your claimed static domain, so the public URL never changes between sessions. **Edit `STATIC_DOMAIN` below before running.**

In [ ]:
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
conf.get_default().auth_token = secrets.get_secret("NGROK_AUTHTOKEN")

# EDIT ME: the exact static domain you claimed in the ngrok dashboard.
STATIC_DOMAIN = "yourname-something.ngrok-free.app"

tunnel = ngrok.connect(8766, domain=STATIC_DOMAIN)
print("Public URL:", tunnel.public_url)
print("Paste this into the extension popup's Local Pipeline URL field:")
print(f"  {tunnel.public_url}/v1/translate-image")

## Cell 7 — keep the session alive
Kaggle interactive sessions can idle-disconnect. This cell keeps the notebook actively running and prints a periodic health check. **This is an intentional infinite loop** -- interrupt it (■ Stop) when you're done reading and want to move to the shutdown cell.

In [ ]:
import time

while True:
    try:
        status = get_json("http://127.0.0.1:8766/v1/health")
        print(
            time.strftime("%H:%M:%S"), "ok, warmup:", status["warmup"]["status"],
            "active jobs:", status["scheduler"]["active"],
        )
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "backend check failed:", e)
    time.sleep(60)

## Cell 8 — clean shutdown
Closes the tunnel, stops the backend, and removes the decoded `.env` from disk so it doesn't linger in `/kaggle/working` if you save outputs.

In [ ]:
import os

ngrok.disconnect(tunnel.public_url)
backend_proc.terminate()
if os.path.exists("/kaggle/working/fmt.env"):
    os.remove("/kaggle/working/fmt.env")
print("Shut down cleanly.")